# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fksifat/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits model trustworthiness, not just model score. I use constructive methodology questions on two paper findings, then re-test my own Week-5 model under an honest split, run a leakage stress test, inspect real errors, and rewrite claims in public-safe language.

## 1. Two paper findings + my methodology questions

### Finding A (paper): model-assisted ranking improves top-of-queue precision versus a rules baseline
**My methodology question:** where exactly does the label come from, and is every feature window strictly earlier than the label window? If the feature window overlaps the outcome window, the measured gain could be partly leakage rather than true forward prediction skill.

### Finding B (paper): exposure, position, and content-age signals are among the strongest drivers
**My methodology question:** does the validation design show this pattern is stable across unseen clients or future periods, or only on random row splits? I would want grouped or time-aware validation, plus per-group variance, before treating this as a decision-support signal that generalizes.

Both questions are respectful by design: they assume good intent and focus on whether the evidence structure fully supports the scope of the claim.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

df = pd.read_csv(FEATURE_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {FEATURE_PATH}")

TARGET = "is_declining_label"
GROUP_COL = "client_id"
ID_COLS = ["content_id", "client_id"]
KNOWN_LEAKAGE = ["trend_direction", "trend_pct", TARGET]

def precision_at_k(y_true, y_score, k=50):
    order = np.argsort(-np.asarray(y_score))
    top_idx = order[: min(k, len(order))]
    return float(np.mean(np.asarray(y_true)[top_idx]))

def build_pipeline(num_cols, cat_cols):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                    ]
                ),
                num_cols,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                cat_cols,
            ),
        ],
        remainder="drop",
    )
    model = RandomForestClassifier(
        n_estimators=180,
        max_depth=10,
        min_samples_leaf=25,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return Pipeline(steps=[("pre", preprocessor), ("model", model)])

def evaluate_split(frame, feature_cols, train_idx, test_idx):
    train = frame.iloc[train_idx].copy()
    test = frame.iloc[test_idx].copy()

    X_train = train[feature_cols]
    X_test = test[feature_cols]
    y_train = train[TARGET].astype(int).to_numpy()
    y_test = test[TARGET].astype(int).to_numpy()

    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(frame[c])]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]

    pipe = build_pipeline(numeric_cols, categorical_cols)
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "rows_test": int(len(test)),
        "base_rate": float(y_test.mean()),
        "precision_at_50": precision_at_k(y_test, y_prob, k=50),
        "average_precision": float(average_precision_score(y_test, y_prob)),
        "roc_auc": float(roc_auc_score(y_test, y_prob)),
        "pred_positive_rate": float(y_pred.mean()),
    }
    pred_frame = test[ID_COLS + [TARGET, "trend_direction", "trend_pct"]].copy()
    pred_frame["y_prob"] = y_prob
    pred_frame["y_pred"] = y_pred
    return metrics, pred_frame, pipe

Loaded 30,000 rows and 52 columns from /home/farhankabirsifat/Desktop/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


## 2. My model under an honest split (before/after)

I compare two validation designs on the same model family and same feature set.

- **Before:** random row split (stratified)
- **After:** grouped split by `client_id` (unseen-client holdout)

The grouped split is closer to deployment reality because rows from the same client can share hidden patterns; random row split can overstate measured generalization.

In [6]:
usable_cols = [
    c
    for c in df.columns
    if c not in (ID_COLS + KNOWN_LEAKAGE + ["provider_used", "model_used", "char_count_tier"])
    and c != TARGET
]

# Random row split (before)
all_idx = np.arange(len(df))
train_idx_random, test_idx_random = train_test_split(
    all_idx,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[TARGET].astype(int),
)

# Grouped split by client_id (after)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx_group, test_idx_group = next(
    gss.split(df, df[TARGET].astype(int), groups=df[GROUP_COL].astype(str))
)

random_metrics, random_pred, _ = evaluate_split(df, usable_cols, train_idx_random, test_idx_random)
group_metrics, grouped_pred, grouped_pipe = evaluate_split(df, usable_cols, train_idx_group, test_idx_group)

comparison = pd.DataFrame([
    {"split": "random_row_split", **random_metrics},
    {"split": "group_client_holdout", **group_metrics},
]).set_index("split")

display(comparison.round(4))
print("\nInterpretation:")
print("- Use grouped result as the more decision-relevant estimate for unseen clients.")
print("- Gap between random and grouped is itself a useful risk signal.")

,rows_test,base_rate,precision_at_50,average_precision,roc_auc,pred_positive_rate
split,,,,,,
random_row_split,6000,0.542,1.00,0.8933,0.8748,0.5753
group_client_holdout,6163,0.511,0.94,0.7883,0.7936,0.5643



Interpretation:
- Use grouped result as the more decision-relevant estimate for unseen clients.
- Gap between random and grouped is itself a useful risk signal.


## 3. Leakage audit

I run a controlled stress test: compare an honest feature set against a deliberately leaky feature set that adds `trend_pct` and `trend_direction` (known siblings of the label definition).

If performance jumps sharply after adding those columns, that is evidence of leakage risk rather than genuine predictive improvement.

In [7]:
leak_candidates = [c for c in ["trend_pct", "trend_direction"] if c in df.columns]

honest_metrics, _, honest_pipe = evaluate_split(df, usable_cols, train_idx_group, test_idx_group)
leaky_cols = usable_cols + leak_candidates
leaky_metrics, _, leaky_pipe = evaluate_split(df, leaky_cols, train_idx_group, test_idx_group)

leakage_compare = pd.DataFrame([
    {"feature_set": "honest", **honest_metrics},
    {"feature_set": "with_leakage_candidates", **leaky_metrics},
]).set_index("feature_set")

display(leakage_compare.round(4))

# Inspect top feature importances from the leaky model to see if suspect columns dominate
pre = leaky_pipe.named_steps["pre"]
model = leaky_pipe.named_steps["model"]
feature_names = pre.get_feature_names_out()
importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)
top_importance = importances.head(12).rename("importance").rename_axis("feature").reset_index()
display(top_importance)

print("\nLeakage audit note:")
if leak_candidates:
    print(f"Tested leakage candidates: {', '.join(leak_candidates)}")
else:
    print("No explicit leakage candidate columns were present.")

,rows_test,base_rate,precision_at_50,average_precision,roc_auc,pred_positive_rate
feature_set,,,,,,
honest,6163,0.511,0.94,0.7883,0.7936,0.5643
with_leakage_candidates,6163,0.511,1.00,1.0000,1.0000,0.5110


,feature,importance
0,num__trend_pct,0.340001
1,cat__trend_direction_down,0.321722
2,cat__trend_direction_stable,0.118384
3,cat__trend_direction_up,0.079449
4,num__impressions_prev_30d,0.034424
5,cat__trend_direction_new,0.015247
6,num__days_with_impressions,0.012174
7,num__impressions_90d,0.010115
8,num__avg_position,0.007038
9,num__log_impressions_90d,0.006725



Leakage audit note:
Tested leakage candidates: trend_pct, trend_direction


## 4. Claim rewrite

### Original bold claim (too strong)
"Our model predicts traffic decline accurately and should be used to choose refresh actions."

### Rewritten claim (safe, evidence-aligned)
"In this dataset, we **observed** that the grouped client-holdout model ranked likely declining pages better than the baseline at the top of the queue. This is a **measured, directional** result and should be used as **decision-support** with human review, not as an automated causal decision rule."

This rewrite narrows scope, states what was measured, and avoids causal overreach.

In [8]:
# Real failure examples from grouped holdout predictions
eval_frame = grouped_pred.copy()
eval_frame["error_type"] = "correct"
eval_frame.loc[(eval_frame[TARGET] == 0) & (eval_frame["y_pred"] == 1), "error_type"] = "false_positive"
eval_frame.loc[(eval_frame[TARGET] == 1) & (eval_frame["y_pred"] == 0), "error_type"] = "false_negative"

fp_examples = (
    eval_frame[eval_frame["error_type"] == "false_positive"]
    .sort_values("y_prob", ascending=False)
    .head(5)
)
fn_examples = (
    eval_frame[eval_frame["error_type"] == "false_negative"]
    .sort_values("y_prob", ascending=True)
    .head(5)
)

print("False positives (predicted decline, observed stable):")
display(fp_examples[["content_id", "client_id", "y_prob", TARGET, "trend_direction", "trend_pct"]])

print("False negatives (predicted stable, observed decline):")
display(fn_examples[["content_id", "client_id", "y_prob", TARGET, "trend_direction", "trend_pct"]])

error_summary = eval_frame["error_type"].value_counts(normalize=True).rename("share")
print("\nError mix on grouped holdout:")
display(error_summary.to_frame().round(4))

False positives (predicted decline, observed stable):


,content_id,client_id,y_prob,is_declining_label,trend_direction,trend_pct
20736,content_41baf0722ad9,client_8527a891e2,0.825830,0,stable,-14.3
2357,content_8f1409b2674e,client_8527a891e2,0.817790,0,stable,-17.9
12332,content_4d9f36001f06,client_8527a891e2,0.817048,0,stable,-17.0
10080,content_35d63627bf3e,client_8527a891e2,0.802823,0,stable,8.7
11061,content_0b47dae0c7f9,client_8527a891e2,0.801097,0,stable,-13.3


False negatives (predicted stable, observed decline):


,content_id,client_id,y_prob,is_declining_label,trend_direction,trend_pct
24864,content_8ab9f1033843,client_e629fa6598,0.211335,1,down,-27.6
11655,content_cea79ef51519,client_f369cb89fc,0.238249,1,down,-35.6
27575,content_284df888e0cb,client_e629fa6598,0.239296,1,down,-25.0
17727,content_82074408a03e,client_e629fa6598,0.242379,1,down,-32.8
22663,content_1794327a4d44,client_e629fa6598,0.248719,1,down,-46.5



Error mix on grouped holdout:


,share
error_type,
correct,0.7023
false_positive,0.1756
false_negative,0.1222


## Self-check

Before submission, I verified the following in this notebook run:

- [x] Two paper findings are named, each with a concrete methodology question
- [x] Before/after validation comparison is shown (random vs grouped client holdout)
- [x] Leakage audit includes a controlled test with known suspect columns
- [x] Real failure examples are shown (false positives and false negatives)
- [x] Claims are rewritten in observed/measured/directional/decision-support language
- [x] Notebook runs top-to-bottom with no errors in this environment
- [ ] Commit this executed notebook under `work/notebooks/` and submit repo URL